# Spark Setup

In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

HOST_IP = "192.168.64.1"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .getOrCreate()
)

Streaming Implementation

In [ ]:

# ==================================================
# Schema
# ==================================================

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])


# ==================================================
# Kafka reader
# ==================================================

def read_topic(topic, source):
    return (
        spark.readStream
        .format("kafka")
        .option(
            "kafka.bootstrap.servers",
            f"{HOST_IP}:9092"
        )
        .option("subscribe", topic)
        .load()
        .selectExpr("CAST(value AS STRING)")
        .withColumn("source", lit(source))
        .select(
            from_json(
                col("value"),
                event_schema
            ).alias("data"),
            col("source")
        )
        .select("data.*", "source")
        .withColumn(
            "event_time",
            to_timestamp(col("timestamp"))
        )
    )


# ==================================================
# Read streams
# ==================================================

camera_a_stream = read_topic(
    "camera-events-A",
    "camera-a"
)

camera_b_stream = read_topic(
    "camera-events-B",
    "camera-b"
)

camera_c_stream = read_topic(
    "camera-events-C",
    "camera-c"
)


# ==================================================
# Combine streams
# ==================================================

combined_stream = (
    camera_a_stream
    .union(camera_b_stream)
    .union(camera_c_stream)
    .withWatermark(
        "event_time",
        "10 minutes"
    )
)


# ==================================================
# Camera metadata
# ==================================================

camera_df = spark.read.csv(
    f"{Path('..')}/data/camera.csv",
    header=True,
    inferSchema=True
)


# ==================================================
# Enrich stream
# ==================================================

events_with_camera = (
    combined_stream
    .join(camera_df, "camera_id")
)


# ==================================================
# Instant violations
# ==================================================

instant_violations = (
    events_with_camera
    .filter(
        col("speed_reading")
        > col("speed_limit")
    )
)


# ==================================================
# Average speed join logic
# ==================================================

start_events = (
    events_with_camera.alias("start")
)

end_events = (
    events_with_camera.alias("end")
)

joined_stream = (
    start_events.join(
        end_events,
        expr("""
            start.car_plate = end.car_plate
            AND start.position < end.position
            AND start.event_time < end.event_time
            AND end.event_time <=
                start.event_time + interval 10 minutes
        """)
    )
)


# ==================================================
# Average speed calculation
# ==================================================

average_violations = (
    joined_stream
    .withColumn(
        "distance_km",
        abs(
            col("end.position")
            - col("start.position")
        )
    )
    .withColumn(
        "travel_time_hours",
        (
            unix_timestamp(
                col("end.event_time")
            )
            -
            unix_timestamp(
                col("start.event_time")
            )
        ) / 3600
    )
    .withColumn(
        "average_speed",
        col("distance_km")
        / col("travel_time_hours")
    )
    .filter(
        col("average_speed")
        > col("end.speed_limit")
    )
)


# ==================================================
# Logger helper
# ==================================================

def log_batch(name):

    def logger(batch_df, batch_id):

        now = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row_count = batch_df.count()

        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id}")
        print(f"Rows received: {row_count}")
        print("=" * 70)

        batch_df.show(
            truncate=False
        )

    return logger


# ==================================================
# Queries
# ==================================================

combined_query = (
    combined_stream.writeStream
    .foreachBatch(
        log_batch("combined_stream processed")
    )
    .outputMode("append")
    .start()
)

instant_query = (
    instant_violations.writeStream
    .foreachBatch(
        log_batch(
            "instant_violations processed"
        )
    )
    .outputMode("append")
    .start()
)

average_query = (
    average_violations.writeStream
    .foreachBatch(
        log_batch(
            "average_violations processed"
        )
    )
    .outputMode("append")
    .start()
)


# ==================================================
# Wait for all queries
# ==================================================

spark.streams.awaitAnyTermination()